# LLaMA-3.1-8B Evaluation

## Purpose
Evaluates LLaMA-8B outputs on 9,160 validation sentences. Computes SARI, BLEU, BERTScore, FKGL and compares against BART v2.

## Key Results
- LLaMA-8B SARI: 36.29, BART v2 SARI: 33.23
- FKGL: LLaMA-8B=11.31, BART v2=16.00, Source=13.03

## Hardware
No GPU needed


In [ ]:
# ============================================================
# MASTER SETUP — Run this first every session (~4 min)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', 'transformers', 'datasets', 'sentencepiece', '-q'])
subprocess.run(['pip', 'install', 'git+https://github.com/feralvam/easse.git', '-q'])
subprocess.run(['pip', 'install', 'groq', '-q'])

import torch, gc, os, json, zipfile, ast, re, time
import pandas as pd
import numpy as np
from collections import Counter
from groq import Groq

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gc.collect()
torch.cuda.empty_cache()

# ── Groq client ──
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"  # ← paste your key
client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Verify key
try:
    resp = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": "Say READY"}],
        max_tokens=5
    )
    print(f"✅ Groq key works: {resp.choices[0].message.content}")
except Exception as e:
    print(f"❌ Groq key failed: {e}")

# ── Load test data ──
test_path = '/content/drive/MyDrive/SimpleText2025/simpletext25_task11_test.json'
with open(test_path) as f:
    test_data = json.load(f)
print(f"✅ Test data loaded: {len(test_data)} sentences")

# ── LLM simplification function ──
SYSTEM_PROMPT = """You are a medical text simplifier. Rewrite the sentence in plain English for a 13-year-old with no medical background.
RULES:
- Output ONLY the rewritten sentence, nothing else
- Always change the wording — never copy input unchanged
- Replace medical jargon with everyday words:
  * RCT → clinical trial
  * odds ratio / OR → chance of
  * heterogeneity → variation between studies
  * glucocorticoids → steroid medicines
  * confidence interval / CI → range of uncertainty
  * I2 → how different the study results were
- Keep sentences short and clear
- Target Grade 8 reading level"""

def simplify_with_llm(sentence, max_retries=3):
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": sentence}
                ],
                max_tokens=150,
                temperature=0.3
            )
            result = resp.choices[0].message.content.strip()
            if result.strip() == sentence.strip():
                resp2 = client.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": f"Rewrite in simple words for a 13-year-old. Do NOT copy the original:\n{sentence}"}
                    ],
                    max_tokens=150,
                    temperature=0.7
                )
                result = resp2.choices[0].message.content.strip()
            return result
        except Exception as e:
            print(f"  Retry {attempt+1}: {e}")
            time.sleep(5)
    return sentence

print(f"\n✅ Setup complete | Device: {device}")
print(f"   All functions ready — run the submission cell now")

Mounted at /content/drive
✅ Groq key works: SET...
✅ Test data loaded: 9160 sentences

✅ Setup complete | Device: cuda
   All functions ready — run the submission cell now


In [ ]:
# ============================================================
# FULL SUBMISSION WITH CHECKPOINTING — resumes from last save
# ============================================================
import time, json, zipfile, os

CHECKPOINT_PATH = '/content/drive/MyDrive/SimpleText2025/outputs/llama_checkpoint.json'
OUT_PATH        = '/content/drive/MyDrive/SimpleText2025/outputs/tokatrons_task11_LLaMA8B.json'
ZIP_PATH        = OUT_PATH.replace('.json', '.zip')
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

test_sentences = [item['complex'] for item in test_data]
total          = len(test_sentences)

# ── Load checkpoint if exists ──
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        checkpoint = json.load(f)
    results    = checkpoint['results']
    start_from = len(results)
    print(f"✅ Resuming from sentence {start_from}/{total}")
else:
    results    = []
    start_from = 0
    print(f"Starting fresh — {total} sentences")

# ── Run from checkpoint ──
start_time = time.time()

for i in range(start_from, total):
    sent = test_sentences[i]
    pred = simplify_with_llm(sent)
    results.append(pred)

    # Save checkpoint every 100 sentences
    if (i + 1) % 100 == 0:
        with open(CHECKPOINT_PATH, 'w') as f:
            json.dump({'results': results}, f)

        elapsed = time.time() - start_time
        done    = i + 1 - start_from
        eta     = (elapsed / done) * (total - i - 1)
        changed = sum(1 for p, s in zip(results, test_sentences[:i+1])
                      if p.strip() != s.strip())
        print(f"  {i+1}/{total} | Changed: {changed} | ETA: {eta/60:.1f} min | Saved ✅")

# ── Final save ──
output = [{
    'pair_id':    item['pair_id'],
    'para_id':    item['para_id'],
    'sent_id':    item['sent_id'],
    'complex':    item['complex'],
    'prediction': pred,
    'run_id':     'tokatrons_task11_LLaMA8B'
} for item, pred in zip(test_data, results)]

with open(OUT_PATH, 'w') as f:
    json.dump(output, f, indent=2)
with zipfile.ZipFile(ZIP_PATH, 'w') as zf:
    zf.write(OUT_PATH, arcname='tokatrons_task11_LLaMA8B.json')

changed = sum(1 for p, s in zip(results, test_sentences) if p.strip() != s.strip())
print(f"\n✅ Done! Changed: {changed}/{total} ({changed/total*100:.1f}%)")
print(f"✅ Submit: {ZIP_PATH}")

✅ Resuming from sentence 8100/9160


In [ ]:
# ============================================================
# EVALUATE LLaMA SUBMISSION — All CLEF 2026 Metrics
# Run after the full submission cell completes
# ============================================================
import subprocess
subprocess.run(['pip', 'install', 'textstat', 'sacrebleu', '-q'])

import torch, numpy as np, time
from easse.sari import corpus_sari
from transformers import AutoTokenizer, AutoModel
import sacrebleu, textstat
import torch.nn.functional as F

# ── Step 1: Generate val set predictions with LLaMA ──
print("Generating val set predictions with LLaMA 8B...")
print("(1472 sentences × 0.5s = ~12 min)\n")

val_sentences = val_df['complex'].tolist()
val_refs      = val_df['simple_text'].tolist()
val_preds_llm = []
failed        = []

for i, sent in enumerate(val_sentences):
    pred = simplify_with_llm(sent)
    val_preds_llm.append(pred)
    if pred.strip() == sent.strip():
        failed.append(i)
    time.sleep(0.5)
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(val_sentences)} done...")

print(f"\n✅ Generated {len(val_preds_llm)} predictions")
print(f"   Unchanged: {len(failed)} ({len(failed)/len(val_sentences)*100:.1f}%)")

# ── Step 2: Filter empty/unchanged for scoring ──
valid = [(p, s, r) for p, s, r in zip(val_preds_llm, val_sentences, val_refs)
         if p.strip() != ""]
preds_c = [x[0] for x in valid]
src_c   = [x[1] for x in valid]
refs_c  = [x[2] for x in valid]
print(f"\nValid predictions: {len(preds_c)}/{len(val_preds_llm)}")

# ── Step 3: SARI ──
sari = corpus_sari(
    orig_sents=src_c,
    sys_sents=preds_c,
    refs_sents=[refs_c]
)
print(f"✅ SARI: {sari:.4f}")

# ── Step 4: BLEU ──
bleu = sacrebleu.corpus_bleu(preds_c, [refs_c])
print(f"✅ BLEU: {bleu.score:.4f}")

# ── Step 5: BERTScore (manual — avoids library bug) ──
print("Computing BERTScore...")
bert_tok   = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model = AutoModel.from_pretrained('bert-base-uncased').to(device)
bert_model.eval()

def mean_pool(token_embeds, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(token_embeds.size()).float()
    return (token_embeds * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def get_embeddings(texts, batch_size=64):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = bert_tok(batch, return_tensors='pt', truncation=True,
                         padding=True, max_length=128).to(device)
        with torch.no_grad():
            out = bert_model(**enc)
        emb = mean_pool(out.last_hidden_state, enc['attention_mask'])
        all_embs.append(F.normalize(emb, dim=-1).cpu())
    return torch.cat(all_embs, dim=0)

pred_embs  = get_embeddings(preds_c)
ref_embs   = get_embeddings(refs_c)
bert_f1    = (pred_embs * ref_embs).sum(dim=-1).mean().item()
print(f"✅ BERTScore F1: {bert_f1:.4f}")

# ── Step 6: FKGL ──
src_fkgl  = np.mean([textstat.flesch_kincaid_grade(s) for s in src_c])
pred_fkgl = np.mean([textstat.flesch_kincaid_grade(p) for p in preds_c])
ref_fkgl  = np.mean([textstat.flesch_kincaid_grade(r) for r in refs_c])
print(f"✅ FKGL computed")

# ── Step 7: Compression + stats ──
src_lens   = [len(s.split()) for s in src_c]
pred_lens  = [len(p.split()) for p in preds_c]
comp_ratio = np.mean([p/s if s > 0 else 1.0 for p, s in zip(pred_lens, src_lens)])
deletions  = sum(1 for p, s in zip(pred_lens, src_lens) if p < s)
identical  = sum(1 for p, s in zip(preds_c, src_c) if p.strip() == s.strip())

# ── Final comparison table ──
print(f"\n{'='*62}")
print(f"METRIC COMPARISON — Val Set ({len(preds_c)} sentences)")
print(f"{'='*62}")
print(f"{'Metric':<25} {'BART v2':>10} {'LLaMA 8B':>10} {'Target'}")
print(f"{'-'*62}")
print(f"{'SARI':<25} {'33.23':>10} {sari:>10.4f}  {'↑ higher better'}")
print(f"{'BLEU':<25} {'6.89':>10} {bleu.score:>10.4f}  {'↑ higher better'}")
print(f"{'BERTScore F1':<25} {'0.634':>10} {bert_f1:>10.4f}  {'↑ higher better'}")
print(f"{'-'*62}")
print(f"{'FKGL (source)':<25} {'13.03':>10} {src_fkgl:>10.2f}  {'input complexity'}")
print(f"{'FKGL (prediction)':<25} {'16.00':>10} {pred_fkgl:>10.2f}  {'↓ lower = simpler'}")
print(f"{'FKGL (reference)':<25} {'8.07':>10} {ref_fkgl:>10.2f}  {'target ~8'}")
print(f"{'-'*62}")
print(f"{'Compression Ratio':<25} {'2.85':>10} {comp_ratio:>10.4f}  {'↓ lower = shorter'}")
print(f"{'Deletion Proportion':<25} {'0.073':>10} {deletions/len(preds_c):>10.4f}  {'fraction made shorter'}")
print(f"{'Identical to Source':<25} {'0.005':>10} {identical/len(preds_c):>10.4f}  {'↓ lower = more changed'}")
print(f"{'='*62}")

# ── Save results for LaTeX table ──
latex_row = (
    f"LLaMA-3.1-8B zero-shot & {sari:.2f} & {bleu.score:.2f} & "
    f"{bert_f1:.3f} & {pred_fkgl:.2f} \\\\"
)
print(f"\nLaTeX table row:")
print(latex_row)
print(f"\nPaste this into Table 1 in your paper to replace the TBD values.")